# Neural-network stability and training duration

Evaluate 20 predetermined seeds; do **not** fit a final model on all data. Each seed combines elections through 2015 with half of 2017 for training, uses the other half of 2017 for validation, and evaluates all 2019 rows only after restoring the validation-selected checkpoint.

The median best epoch is the candidate fixed training duration for a later experiment. Variability here combines random initialisation, mini-batch order, and the changing 2017 split; it does not isolate initialisation alone.

Requirements: pandas, NumPy, SciPy, scikit-learn >= 1.2, PyTorch, and a Jupyter Python kernel. Run cells in order. CPU execution is intentional for simple reproducibility; results can still differ across library versions/platforms.

In [ ]:
from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

SEEDS = list(range(42, 62))  # 20 fixed, distinct seeds; not selected by performance
BATCH_SIZE = 64
MAX_EPOCHS = 1500
PATIENCE = 20
MIN_DELTA = 0.001
LEARNING_RATE = 0.001
HIDDEN_SIZES = (64, 32)
DEVICE = torch.device("cpu")
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

print({"numpy": np.__version__, "pandas": pd.__version__,
       "scipy": scipy.__version__, "sklearn": sklearn.__version__,
       "torch": torch.__version__, "device": str(DEVICE)})

## Read the data and define predictors

The inspected CSV uses **election** and **winner**, with classes con, lab, lib, natSW, and oth. It currently contains 632 rows in each of 2017 and 2019. Several previous-election predictors have missing values.

Use the same 14 predictor names as Models/logistic_regression.py, listed explicitly so this experiment remains readable. Unlike that model's filtering, retain missing-predictor rows and the oth class. Exclude current-election vote shares, majority, identifiers, and election year from predictors. No feature selection uses 2019 outcomes. This assumes the upstream polling and projected-share features were constructed using information available before each election.

Inspect development data below; hold 2019 aside until evaluation.

In [ ]:
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "TEST_TRAIN" / "train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the project root or a folder beneath it.")

data = pd.read_csv(PROJECT_ROOT / "TEST_TRAIN" / "train.csv")
YEAR, TARGET = "election", "winner"
FEATURES = [
    "country/region", "previous_majority_proportion", "previous_winner",
    "Conservative", "Labour", "LD", "incumbent",
    "previous_con_share", "previous_lib_share", "previous_lab_share",
    "previous_natSW_share", "projected_con_share",
    "projected_lib_share", "projected_lab_share",
]
required = [YEAR, TARGET, *FEATURES]
missing = set(required) - set(data.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

years = pd.to_numeric(data[YEAR], errors="raise")
if years.isna().any():
    raise ValueError("Election years must not be missing.")
historical = data.loc[years <= 2015].copy()
election_2017 = data.loc[years == 2017].copy()
test_2019 = data.loc[years == 2019].copy()
assert not historical.empty and not election_2017.empty and not test_2019.empty
development = pd.concat([historical, election_2017])
if development[TARGET].isna().any():
    raise ValueError("Training/validation targets must not be missing.")

numeric_columns = historical[FEATURES].select_dtypes(include="number").columns.tolist()
categorical_columns = [c for c in FEATURES if c not in numeric_columns]
display(pd.DataFrame({
    "dtype": development[FEATURES].dtypes.astype(str),
    "missing_development": development[FEATURES].isna().sum(),
}))
display(pd.crosstab(development[YEAR], development[TARGET]))
print("Numeric:", numeric_columns)
print("Categorical:", categorical_columns)

# Historical training rows contain all five classes; never learn classes from 2019.
label_encoder = LabelEncoder().fit(historical[TARGET])
unknown_validation = set(election_2017[TARGET]) - set(label_encoder.classes_)
if unknown_validation:
    raise ValueError(f"2017 classes absent from historical training: {unknown_validation}")
print("Training-derived class mapping:", dict(enumerate(label_encoder.classes_)))

## Preprocessing and model

For each seed, fit a fresh ColumnTransformer on that seed's training rows only. Numeric predictors receive median imputation and standardisation (to aid neural-network optimisation). Categorical predictors receive a missing-value category and one-hot encoding with unknown categories ignored.

The network has hidden widths 64 and 32 with ReLU, and a raw-logit output for CrossEntropyLoss; there is no Softmax layer. Use Adam with learning rate 0.001. These choices stay fixed across seeds.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]), numeric_columns),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(
                strategy="constant", fill_value="__MISSING__", keep_empty_features=True
            )),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
    ])


def as_features(matrix):
    if sparse.issparse(matrix):
        matrix = matrix.toarray()
    array = np.asarray(matrix, dtype=np.float32)
    if not np.isfinite(array).all():
        raise ValueError("Preprocessed predictors contain non-finite values.")
    return torch.from_numpy(array).to(DEVICE)


def as_targets(series):
    return torch.tensor(
        label_encoder.transform(series), dtype=torch.long, device=DEVICE
    )


def make_model(input_width):
    return nn.Sequential(
        nn.Linear(input_width, HIDDEN_SIZES[0]),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZES[0], HIDDEN_SIZES[1]),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZES[1], len(label_encoder.classes_)),
    ).to(DEVICE)

## Train each seed and restore its best checkpoint

An improvement means loss is **at least 0.001 lower** than the best accepted loss. Save that epoch's weights, reset patience, and stop after 20 consecutive epochs without a qualifying improvement (or at 1500). Thus best_epoch refers to the accepted checkpoint, not the stopping epoch; a smaller reduction below min_delta does not replace it.

Validation loss is calculated across the entire validation set after each complete epoch. The 2019 features and labels are first transformed/accessed for evaluation after restoring the checkpoint. No 2019 score affects training.

In [ ]:
def run_seed(seed):
    seed_everything(seed)
    counts = election_2017[TARGET].value_counts()
    smallest_half = len(election_2017) // 2
    can_stratify = counts.min() >= 2 and smallest_half >= len(counts)
    if not can_stratify:
        warnings.warn(f"Seed {seed}: class counts do not permit stratification.")
    train_2017, validation = train_test_split(
        election_2017,
        test_size=0.5,
        random_state=seed,
        stratify=election_2017[TARGET] if can_stratify else None,
    )
    training = pd.concat([historical, train_2017])
    assert set(training.index).isdisjoint(validation.index)
    assert set(training.index).isdisjoint(test_2019.index)
    assert set(validation.index).isdisjoint(test_2019.index)
    assert len(train_2017) + len(validation) == len(election_2017)

    preprocessor = make_preprocessor()
    x_train = as_features(preprocessor.fit_transform(training[FEATURES]))
    x_validation = as_features(preprocessor.transform(validation[FEATURES]))
    y_train = as_targets(training[TARGET])
    y_validation = as_targets(validation[TARGET])

    loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
    )
    model = make_model(x_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_loss = criterion(model(x_validation), y_validation).item()
        if not np.isfinite(validation_loss):
            raise RuntimeError(f"Non-finite validation loss for seed {seed}, epoch {epoch}")

        if best_loss - validation_loss >= MIN_DELTA:
            best_loss = validation_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    stopping_epoch = epoch
    assert best_state is not None and 1 <= best_epoch <= stopping_epoch
    model.load_state_dict(best_state)
    model.eval()

    # First use of test predictors/targets: evaluation of the frozen checkpoint.
    x_test = as_features(preprocessor.transform(test_2019[FEATURES]))
    y_test = as_targets(test_2019[TARGET])
    with torch.no_grad():
        predictions = model(x_test).argmax(dim=1)
        accuracy = (predictions == y_test).float().mean().item()
        restored_loss = criterion(model(x_validation), y_validation).item()
    assert len(predictions) == len(test_2019)
    assert np.isclose(restored_loss, best_loss), "Checkpoint restoration failed."

    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_validation_loss": best_loss,
        "stopping_epoch": stopping_epoch,
        "2019_accuracy": accuracy,
    }

In [ ]:
records = []
for seed in SEEDS:
    result = run_seed(seed)
    records.append(result)
    print(
        f"Seed {seed}: best epoch {result['best_epoch']}, "
        f"stopped {result['stopping_epoch']}, "
        f"validation loss {result['best_validation_loss']:.4f}"
    )

results = pd.DataFrame(records)
assert len(results) == len(SEEDS) == results["seed"].nunique()
assert results["2019_accuracy"].between(0, 1).all()
display(results)

## Summarise stability and candidate training duration

Report sample standard deviations (ddof=1). Accuracy is shown as a percentage; its standard deviation is in percentage points. Use the median best epoch, not the epoch or seed with the highest 2019 accuracy. A fractional median can be rounded up for a later fixed-duration run.

These are descriptive results across splits/initialisations on the same test election, not uncertainty across future elections. The median is only a candidate: training on a larger dataset changes the number of updates per epoch. This notebook does not retrain a final model.

In [ ]:
epochs = results["best_epoch"]
accuracies = results["2019_accuracy"]
median_best_epoch = epochs.median()
candidate_fixed_epochs = int(np.ceil(median_best_epoch))

print(f"Number of seeds evaluated: {len(results)}")
print(f"Median best epoch (main quantity): {median_best_epoch:.1f}")
print(f"Mean best epoch: {epochs.mean():.2f}")
print(f"Standard deviation of best epochs: {epochs.std(ddof=1):.2f}")
print(f"Minimum best epoch: {epochs.min()}")
print(f"Maximum best epoch: {epochs.max()}")
print(f"Mean 2019 accuracy: {accuracies.mean():.2%}")
print(f"Standard deviation of 2019 accuracy: {100 * accuracies.std(ddof=1):.2f} percentage points")
print(f"Minimum 2019 accuracy: {accuracies.min():.2%}")
print(f"Maximum 2019 accuracy: {accuracies.max():.2%}")
print(f"Candidate fixed epochs (median rounded up): {candidate_fixed_epochs}")

if results["stopping_epoch"].eq(MAX_EPOCHS).any():
    print("Some runs reached the epoch cap; inspect those runs before interpreting the duration.")

display(results)